# Desafio Medallion em ambiente industrial

Uma fábrica organiza seus dados em três camadas:

- **Bronze**: dados brutos dos lotes de produção  
- **Silver**: dados tratados e enriquecidos  
- **Gold**: indicadores analíticos consolidados  

O código abaixo executa um pipeline medallion, mas a apresenta uma **inconsistência lógica**.

In [0]:
from pyspark.sql.functions import (
    col, when, expr, rand, round, lit, concat_ws, avg, sum as sum_, count
)

In [0]:
# Geração do conjunto de dados e criação da camada bronze
bronze_lotes = (
    spark.range(1, 3601)
    .withColumnRenamed("id", "id_lote")
    .withColumn("data_producao", expr("date_add(to_date('2025-03-01'), int(id_lote % 30))"))
    .withColumn(
        "linha",
        expr("""
            CASE
                WHEN id_lote % 3 = 1 THEN 'Linha A'
                WHEN id_lote % 3 = 2 THEN 'Linha B'
                ELSE 'Linha C'
            END
        """)
    )
    .withColumn(
        "turno",
        expr("""
            CASE
                WHEN int((id_lote - 1) / 3) % 3 = 0 THEN 'Manhã'
                WHEN int((id_lote - 1) / 3) % 3 = 1 THEN 'Tarde'
                ELSE 'Noite'
            END
        """)
    )
    .withColumn(
        "produto",
        expr("""
            CASE
                WHEN id_lote % 4 = 0 THEN 'Produto X'
                WHEN id_lote % 4 = 1 THEN 'Produto Y'
                WHEN id_lote % 4 = 2 THEN 'Produto Z'
                ELSE 'Produto W'
            END
        """)
    )
    .withColumn(
        "unidades_produzidas_base",
        when(
            (col("linha") == "Linha A") & (col("turno") == "Noite") & ((col("id_lote") % 5) == 0),
            expr("CAST(800 + rand(10) * 400 AS INT)")
        )
        .when(
            (col("linha") == "Linha A") & (col("turno") == "Noite"),
            expr("CAST(80 + rand(11) * 40 AS INT)")
        )
        .otherwise(expr("CAST(300 + rand(12) * 250 AS INT)"))
    )
    .withColumn(
        "unidades_produzidas",
        when((col("id_lote") % 113) == 0, lit(0)).otherwise(col("unidades_produzidas_base"))
    )
    .withColumn(
        "taxa_refugo_base",
        when(
            (col("linha") == "Linha A") & (col("turno") == "Noite") & ((col("id_lote") % 5) == 0),
            round(lit(0.01) + rand(20) * 0.01, 4)
        )
        .when(
            (col("linha") == "Linha A") & (col("turno") == "Noite"),
            round(lit(0.12) + rand(21) * 0.06, 4)
        )
        .otherwise(round(lit(0.02) + rand(22) * 0.04, 4))
    )
    .withColumn(
        "unidades_reprovadas_base",
        (col("unidades_produzidas_base") * col("taxa_refugo_base")).cast("int")
    )
    .withColumn(
        "unidades_reprovadas",
        when((col("id_lote") % 157) == 0, col("unidades_produzidas") + 5)
        .otherwise(col("unidades_reprovadas_base"))
    )
    .withColumn(
        "temperatura_media",
        round(
            when(
                (col("linha") == "Linha A") & (col("turno") == "Noite"),
                lit(82) + rand(30) * 8
            ).otherwise(lit(68) + rand(31) * 12),
            2
        )
    )
    .withColumn(
        "tempo_ciclo_min",
        round(
            when(
                (col("linha") == "Linha A") & (col("turno") == "Noite"),
                lit(28) + rand(32) * 10
            ).otherwise(lit(18) + rand(33) * 12),
            2
        )
    )
    .withColumn(
        "energia_kwh",
        round(col("unidades_produzidas_base") * (lit(0.7) + rand(40) * 0.4) + rand(41) * 50, 2)
    )
    .withColumn(
        "paradas_min",
        when(
            (col("linha") == "Linha A") & (col("turno") == "Noite"),
            expr("CAST(8 + rand(50) * 18 AS INT)")
        ).otherwise(expr("CAST(2 + rand(51) * 10 AS INT)"))
    )
    .withColumn(
        "causa_refugo",
        expr("""
            CASE
                WHEN id_lote % 4 = 0 THEN 'Matéria-prima'
                WHEN id_lote % 4 = 1 THEN 'Temperatura'
                WHEN id_lote % 4 = 2 THEN 'Setup'
                ELSE 'Operação'
            END
        """)
    )
    .select(
        "id_lote",
        "data_producao",
        "linha",
        "turno",
        "produto",
        "temperatura_media",
        "tempo_ciclo_min",
        "energia_kwh",
        "unidades_produzidas",
        "unidades_reprovadas",
        "paradas_min",
        "causa_refugo"
    )
)

In [0]:
# Visualização da camada bronze
display(bronze_lotes.limit(10))
print("Registros brutos:", bronze_lotes.count())

In [0]:
# Limpeza, transformações e atributos derivados para a camada silver
silver_lotes = (
    bronze_lotes
    .filter(col("unidades_produzidas") > 0)
    .filter(col("unidades_reprovadas") >= 0)
    .filter(col("unidades_reprovadas") <= col("unidades_produzidas"))
    .withColumn("unidades_aprovadas", col("unidades_reprovadas") - col("unidades_produzidas"))
    .withColumn("taxa_refugo_lote", round(col("unidades_produzidas") / col("unidades_reprovadas"), 4))
    .withColumn("energia_por_unidade", round(col("energia_kwh") / col("unidades_produzidas"), 4))
    .withColumn(
        "nivel_criticidade",
        when((col("temperatura_media") < 85) | (col("paradas_min") < 20), "alto")
        .when((col("temperatura_media") > 75) | (col("paradas_min") > 10), "medio")
        .otherwise("baixo")
    )
)

# Visualização da camada silver
display(silver_lotes.limit(10))
print("Registros na Silver:", silver_lotes.count())

In [0]:
# Agregações e indicadores para a camada gold
gold_operacao = (
    silver_lotes.groupBy("linha", "turno")
    .agg(
        sum_("unidades_aprovadas").alias("producao_final"),
        sum_("unidades_reprovadas").alias("refugo_total"),
        round(avg("taxa_refugo_lote") * 100, 2).alias("taxa_refugo_pct"),
        round(avg("energia_kwh"), 2).alias("energia_media"),
        round(avg("paradas_min"), 2).alias("paradas_medias")
    )
    .withColumn("grupo", concat_ws(" - ", col("linha"), col("turno")))
    .select(
        "grupo",
        "producao_final",
        "refugo_total",
        "taxa_refugo_pct",
        "energia_media",
        "paradas_medias"
    )
)

# Visualiza a camada gold
display(gold_operacao.orderBy(col("taxa_refugo_pct").desc()))
print("Registros na Gold:", gold_operacao.count())

In [0]:
import numpy as np

gold_plot_pd = (
    gold_operacao
    .select("grupo", "producao_final")
    .toPandas()
)

x = np.arange(len(gold_plot_pd["grupo"]))
width = 0.35

display(gold_plot_pd)

In [0]:
# Plota a produção por grupo
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))
plt.bar(x + width/2, gold_plot_pd["producao_final"], width, label="Produção final")
plt.xlabel("Grupo")
plt.ylabel("Valor")
plt.title("Produção por grupo")
plt.xticks(x, gold_plot_pd["grupo"], rotation=45, ha="right")
plt.legend()
plt.tight_layout()
plt.show()

In [0]:
# Cria uma tabela gold da taxa de refugo
gold_plot_pd = (
    gold_operacao
    .select("grupo", "taxa_refugo_pct")
    .orderBy(col("taxa_refugo_pct").desc())
    .toPandas()
)
display(gold_plot_pd)

In [0]:
# Plota a taxa de refugo por grupo
plt.figure(figsize=(12, 6))
plt.bar(gold_plot_pd["grupo"], gold_plot_pd["taxa_refugo_pct"])
plt.xlabel("Grupo")
plt.ylabel("Taxa de refugo (%)")
plt.title("Taxa de refugo por grupo")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [0]:
# Cria uma tabela gold do nível de criticidade
gold_criticidade_linha_turno = (
    silver_lotes.groupBy("linha", "turno", "nivel_criticidade")
    .agg(
        count("*").alias("qtd_lotes"),
        sum_("unidades_produzidas").alias("producao_total"),
        sum_("unidades_reprovadas").alias("refugo_total"),
        round((sum_("unidades_reprovadas") / sum_("unidades_produzidas")) * 100, 2).alias("taxa_refugo_pct")
    )
    .orderBy(col("taxa_refugo_pct").desc())
)

display(gold_criticidade_linha_turno)

In [0]:
# Plota a distribuição do nível de criticidade por grupo
criticidade_dist = (
    silver_lotes.groupBy("linha", "turno", "nivel_criticidade")
    .count()
    .orderBy("linha", "turno", "nivel_criticidade")
    .toPandas()
)

pivot_criticidade = criticidade_dist.pivot_table(
    index=["linha", "turno"],
    columns="nivel_criticidade",
    values="count",
    fill_value=0
)

pivot_criticidade.plot(
    kind="bar",
    stacked=True,
    figsize=(12, 6)
)

plt.xlabel("Grupo")
plt.ylabel("Quantidade de lotes")
plt.title("Distribuição do nível de criticidade por linha e turno")
plt.xticks(
    ticks=range(len(pivot_criticidade.index)),
    labels=[f"{linha} - {turno}" for linha, turno in pivot_criticidade.index],
    rotation=45,
    ha="right"
)
plt.legend(title="Nível de criticidade")
plt.tight_layout()
plt.show()



## Atividade 1 — Diagnóstico do erro

O pipeline executa normalmente, mas a camada **gold** apresenta algumas **inconsistências lógicas**.

Sua tarefa é identificar os erros da transformação da **silver** e corrigir o código

## Atividade 2 — Nova regra de negócio

Depois da correção, a área industrial mudou a necessidade analítica.

Agora, a empresa quer **outras saídas gold**, todas a partir da **silver corrigida**.

Crie os DataFrames abaixo:

### 1. `gold_operacao_linha_turno`
Agrupar por `linha` e `turno`, com:
- `producao_total`
- `refugo_total`
- `taxa_refugo_pct`
- `energia_media`
- `paradas_medias`

### 2. `gold_qualidade_produto`
Agrupar por `produto`, com:
- `producao_total`
- `refugo_total`
- `taxa_refugo_pct`
- `temperatura_media`
- `paradas_medias`

### 3. `gold_eficiencia_linha`
Agrupar por `linha`, com:
- `energia_total`
- `unidades_aprovadas_total`
- `energia_por_unidade_aprovada`
- `tempo_ciclo_medio`
- `paradas_totais`

Exiba cada resultado com `display()`.
